# 01.4 · Pandas: selecting, filtering, sorting, new columns

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/04-pandas-select-filter/lesson.ipynb)

Read the [README](README.md) first. Here you reproduce the by-hand answers on six rows, then answer marketing, finance and operations on the real table.

## 1. Six order lines

In [1]:
import pandas as pd

df = pd.DataFrame({
    "Invoice":     ["536365", "536367", "536370", "536370", "C536379", "536381"],
    "Description": ["WHITE HANGING HEART", "ASSORTED COLOUR BIRD", "RED TOADSTOOL LED",
                    "ALARM CLOCK BAKELIKE", "DISCOUNT", "PINK HEART SHAPE"],
    "Quantity":    [6, 32, 24, 48, -1, 12],
    "Price":       [2.55, 1.69, 1.65, 3.75, 27.50, 1.25],
    "Country":     ["United Kingdom", "France", "Germany", "Germany", "Germany", "United Kingdom"],
})
df

,Invoice,Description,Quantity,Price,Country
0,536365,WHITE HANGING HEART,6,2.55,United Kingdom
1,536367,ASSORTED COLOUR BIRD,32,1.69,France
2,536370,RED TOADSTOOL LED,24,1.65,Germany
3,536370,ALARM CLOCK BAKELIKE,48,3.75,Germany
4,C536379,DISCOUNT,-1,27.50,Germany
5,536381,PINK HEART SHAPE,12,1.25,United Kingdom


## 2. Which columns?

Single brackets with a name give a **Series**. Double brackets (a list inside) give a **DataFrame**.

In [2]:
print(type(df["Price"]), "   ← Series")
print(type(df[["Price"]]), "← DataFrame")
df[["Invoice", "Description"]]

<class 'pandas.Series'>    ← Series
<class 'pandas.DataFrame'> ← DataFrame


,Invoice,Description
0,536365,WHITE HANGING HEART
1,536367,ASSORTED COLOUR BIRD
2,536370,RED TOADSTOOL LED
3,536370,ALARM CLOCK BAKELIKE
4,C536379,DISCOUNT
5,536381,PINK HEART SHAPE


## 3. Which rows? A mask

Finance first: add `revenue`, because every other request needs it.

In [3]:
df["revenue"] = df["Quantity"] * df["Price"]
df

,Invoice,Description,Quantity,Price,Country,revenue
0,536365,WHITE HANGING HEART,6,2.55,United Kingdom,15.30
1,536367,ASSORTED COLOUR BIRD,32,1.69,France,54.08
2,536370,RED TOADSTOOL LED,24,1.65,Germany,39.60
3,536370,ALARM CLOCK BAKELIKE,48,3.75,Germany,180.00
4,C536379,DISCOUNT,-1,27.50,Germany,-27.50
5,536381,PINK HEART SHAPE,12,1.25,United Kingdom,15.00


By hand: 15.30, 54.08, 39.60, 180.00, −27.50, 15.00. Now the masks.

In [4]:
is_germany = df["Country"] == "Germany"
print(is_germany.to_numpy())
df[is_germany]

[False False  True  True  True False]


,Invoice,Description,Quantity,Price,Country,revenue
2,536370,RED TOADSTOOL LED,24,1.65,Germany,39.6
3,536370,ALARM CLOCK BAKELIKE,48,3.75,Germany,180.0
4,C536379,DISCOUNT,-1,27.50,Germany,-27.5


In [5]:
df[is_germany & (df["revenue"] > 50)]          # brackets around each condition; & not and

,Invoice,Description,Quantity,Price,Country,revenue
3,536370,ALARM CLOCK BAKELIKE,48,3.75,Germany,180.0


## 4. Rows *and* columns: `.loc`

`.loc[rows, columns]`: the mask picks rows, the list picks columns. This is marketing's request in one line.

In [6]:
df.loc[is_germany & (df["revenue"] > 50), ["Invoice", "Description", "revenue"]]

,Invoice,Description,revenue
3,536370,ALARM CLOCK BAKELIKE,180.0


### `.loc` is by label, `.iloc` is by position

With the default 0…5 index they look alike, until you slice.

In [7]:
print("iloc[1:3] → positions 1, 2       :", df.iloc[1:3]["Country"].tolist())
print("loc[1:3]  → labels 1, 2 AND 3    :", df.loc[1:3]["Country"].tolist())

iloc[1:3] → positions 1, 2       : ['France', 'Germany']
loc[1:3]  → labels 1, 2 AND 3    : ['France', 'Germany', 'Germany']


## 5. Masks for the common cases

In [8]:
print("isin       :", df["Country"].isin(["France", "Germany"]).to_numpy())
print("between    :", df["revenue"].between(15, 40).to_numpy())          # inclusive
print("startswith :", df["Invoice"].str.startswith("C").to_numpy())
print("contains   :", df["Description"].str.contains("HEART").to_numpy())

isin       : [False  True  True  True  True False]
between    : [ True False  True False False  True]
startswith : [False False False False  True False]
contains   : [ True False False False False  True]


In [9]:
df[df["Invoice"].str.startswith("C")]           # operations: cancelled lines

,Invoice,Description,Quantity,Price,Country,revenue
4,C536379,DISCOUNT,-1,27.5,Germany,-27.5


## 6. Sorting and top-n

In [10]:
df.sort_values("revenue", ascending=False)

,Invoice,Description,Quantity,Price,Country,revenue
3,536370,ALARM CLOCK BAKELIKE,48,3.75,Germany,180.00
1,536367,ASSORTED COLOUR BIRD,32,1.69,France,54.08
2,536370,RED TOADSTOOL LED,24,1.65,Germany,39.60
0,536365,WHITE HANGING HEART,6,2.55,United Kingdom,15.30
5,536381,PINK HEART SHAPE,12,1.25,United Kingdom,15.00
4,C536379,DISCOUNT,-1,27.50,Germany,-27.50


By hand: 180.00, 54.08, 39.60, 15.30, 15.00, −27.50 → rows 3, 1, 2, 0, 5, 4. `sort_values` returns a **new** frame; `df` itself is unchanged.

In [11]:
print(df.sort_values(["Country", "revenue"], ascending=[True, False])[["Country", "revenue"]])
print()
print(df.nlargest(2, "revenue")[["Description", "revenue"]])

          Country  revenue
1          France    54.08
3         Germany   180.00
2         Germany    39.60
4         Germany   -27.50
0  United Kingdom    15.30
5  United Kingdom    15.00

            Description  revenue
3  ALARM CLOCK BAKELIKE   180.00
1  ASSORTED COLOUR BIRD    54.08


## 7. The whole request as one chain

Filter → derive → filter → sort → select. Reads top to bottom like the English request.

In [12]:
(df[df["Country"] == "Germany"]
   .assign(revenue=lambda d: d["Quantity"] * d["Price"])
   .query("revenue > 50")
   .sort_values("revenue", ascending=False)
   [["Invoice", "Description", "revenue"]])

,Invoice,Description,revenue
3,536370,ALARM CLOCK BAKELIKE,180.0


## 8. The warning everyone meets once

Filter, then assign to the result: Pandas cannot tell whether you meant to change the original. Make a `.copy()` and the ambiguity is gone.

In [13]:
import warnings
warnings.simplefilter("ignore")        # just so the lesson output stays readable

germany = df[df["Country"] == "Germany"]          # a view? a copy? Pandas is not sure
germany = df[df["Country"] == "Germany"].copy()   # now it is certainly a copy
germany["note"] = "EU"
print(germany[["Country", "note"]])
print("original untouched:", "note" in df.columns)

   Country note
2  Germany   EU
3  Germany   EU
4  Germany   EU
original untouched: False


In [14]:
df.loc[df["Country"] == "Germany", "region"] = "EU"     # writing to the original: always .loc
df[["Country", "region"]]

,Country,region
0,United Kingdom,NaN
1,France,NaN
2,Germany,EU
3,Germany,EU
4,Germany,EU
5,United Kingdom,NaN


## 9. On real data

Load the table (the fast CSV copy from 01.3), then answer the three requests.

In [15]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")
if csv.exists():
    retail = pd.read_csv(csv, parse_dates=["InvoiceDate"])
else:
    retail = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    retail.to_csv(csv, index=False)
retail["Invoice"] = retail["Invoice"].astype(str)        # some invoices start with "C": keep all as text
retail["StockCode"] = retail["StockCode"].astype(str)
print(retail.shape)

(541910, 8)


In [16]:
# Finance: a permanent revenue column
retail["revenue"] = retail["Quantity"] * retail["Price"]

# Marketing: German lines over £50, biggest first
germany_big = (retail
    .loc[(retail["Country"] == "Germany") & (retail["revenue"] > 50),
         ["Invoice", "Description", "Quantity", "Price", "revenue"]]
    .sort_values("revenue", ascending=False))
print(f"{len(germany_big):,} German lines over £50")
germany_big.head(10)

806 German lines over £50


,Invoice,Description,Quantity,Price,revenue
378279,569640,REGENCY CAKESTAND 3 TIER,80,10.95,876.00
534968,581179,REGENCY CAKESTAND 3 TIER,64,10.95,700.80
410820,572134,STOOL HOME SWEET HOME,60,10.40,624.00
479551,577168,REGENCY CAKESTAND 3 TIER,56,10.95,613.20
398148,571223,Manual,1,599.50,599.50
455645,575632,Manual,1,549.34,549.34
56218,541093,REGENCY CAKESTAND 3 TIER,48,10.95,525.60
308321,563950,REGENCY CAKESTAND 3 TIER,48,10.95,525.60
187584,552980,POSTAGE,1,523.00,523.00
455644,575632,Manual,1,424.06,424.06


In [17]:
# Operations: cancelled invoices
cancelled = retail[retail["Invoice"].str.startswith("C")]
print(f"cancelled lines    : {len(cancelled):,}")
print(f"cancelled invoices : {cancelled['Invoice'].nunique():,}")
print(f"value cancelled    : £{cancelled['revenue'].sum():,.0f}")

cancelled lines    : 9,288
cancelled invoices : 3,836
value cancelled    : £-896,812


In [18]:
# Marketing follow-up: HEART products by units sold in the UK (non-cancelled)
heart = retail[
    retail["Description"].str.contains("HEART", na=False)
    & (retail["Country"] == "United Kingdom")
    & ~retail["Invoice"].str.startswith("C")
]
heart.nlargest(5, "Quantity")[["Description", "Quantity", "Price", "revenue"]]

,Description,Quantity,Price,revenue
52709,WHITE HANGING HEART T-LIGHT HOLDER,1930,2.55,4921.50
160542,WHITE HANGING HEART T-LIGHT HOLDER,1930,2.40,4632.00
253989,72 SWEETHEART FAIRY CAKE CASES,1440,0.40,576.00
52133,HEART OF WICKER SMALL,1356,1.93,2617.08
86505,PACK OF 12 HEARTS DESIGN TISSUES,1296,0.22,285.12


`na=False` tells `contains` to treat the ~1,500 missing descriptions as "no match" instead of raising an error. The `~` flips the cancellation mask: *not* cancelled.

The question "which HEART product sold most units **in total**" needs the lines added up per product, which is `groupby`, lesson 01.6.

## Quiz answers

1. `df["Price"]` is a Series (1-D, one column's values); `df[["Price"]]` is a DataFrame with one column (2-D). Models take a DataFrame as `X`.
2. `df["Country"].isin(["United Kingdom", "France"])`, or `(df["Country"] == "United Kingdom") | (df["Country"] == "France")`.
3. `df.loc[2:4]` returns rows labelled 2, 3, 4 (three rows); `df.iloc[2:4]` returns positions 2 and 3 (two rows).
4. Make the filtered result an explicit copy, `sub = df[mask].copy()`, before adding columns; or write into the original with `df.loc[mask, "col"] = value`.
5. Some invoice numbers start with the letter `C` (cancellations), so the column cannot be stored as integers; Pandas keeps it as text (or mixed, which we force to text with `astype(str)`).

**Next:** [01.5 Pandas: missing values, duplicates, data types, dates](../05-pandas-clean/)